# SPAR pilot 1: auditable untuned baselines

**Purpose:** establish the measurement instrument before interpreting what selective fine-tuning changes.
We evaluate `Qwen/Qwen2.5-1.5B-Instruct` on fixed GSM8K and MMLU subsets, save every item,
and keep enough provenance to reuse the evaluation on a later LoRA adapter.

**First run:** run sections 1–7, inspect the audit, then explicitly approve it in section 8.
Continue to the full evaluation and results. Do not use **Run all** on your first visit.
This notebook calls the repository CLI; it does not implement a second scorer.

**Validation status:** the harness passed 62 CPU tests before this notebook was authored.
The notebook's Python cells were checked locally, but real Colab/GPU execution still needs the audit below.

## Where this fits into SPAR Team 2

The broader project asks **how much, where along depth, and in which direction fine-tuning writes into a model**,
with comparisons between SFT and RL objectives and between starting models.

| Stage | Work | This notebook's role |
|---|---|---|
| Measurement setup | Freeze prompts, data, scorers and artifacts | Establish and audit the baseline instrument |
| B0 / collapse gate | Train matched rank-1 SFT and GRPO adapters; evaluate them | Reuse these item IDs and metrics on each adapter |
| Modules 7 and 9 | Reproduce SFT collapse; test steering or RL recovery | Supply baseline accuracy and response length for paired comparisons |
| Retention checks | Evaluate MMLU under careful scoring | Keep scoring and prompting differences visible |
| Activation analyses | Measure effective writes, alignment and causal interventions | Connect later behavioral measurements to activation measurements |

We do **not** train adapters, measure activations, establish a causal mechanism, or run GRPO here.
Later adapter evaluation can use this same CLI, but training-data overlap, adapter pairing and sampled-decoding
aggregation need their own recorded protocol. One baseline run is not a multi-seed scientific result.

## What we run and what the earlier numbers mean

“Untuned” means the **instruction-tuned model before this project's additional fine-tuning**, not Qwen's base model.

| Cell | Items | Prompt | Measurement |
|---|---:|---|---|
| GSM8K | 150 official test items | Step-by-step solution; final `#### <number>` | Strict and flexible accuracy; mean/median response length |
| MMLU text, 0-shot | 1,140: 20 per each of 57 subjects | No demonstrations | Greedily generated A/B/C/D text |
| MMLU text, 5-shot | Same 1,140 | Five same-subject dev demonstrations | Greedily generated A/B/C/D text |
| MMLU logits, 0-shot | Same 1,140 | No demonstrations | Next-token argmax over ` A`, ` B`, ` C`, ` D` |
| MMLU logits, 5-shot | Same 1,140 | Five same-subject dev demonstrations | **Primary MMLU metric** |

Historical SPAR references: GSM8K accuracy **0.640**, mean length **288 tokens**, MMLU accuracy **0.570**.
The earlier pilot also reported SFT falling to 0.420 / 77 tokens. These are observations to investigate,
not results produced by this notebook. Its original items, prompts and scorer are unavailable.
We use a **new versioned protocol** and report gaps without declaring failure from a gap.
The MMLU cohort is subject-balanced and sampled; it is not the full-test or model-card MMLU protocol.

## A short glossary

- **Greedy decoding:** at each step take the token with the highest model score (logit). Here `do_sample=False`, one beam, one output.
- **Sampled decoding:** draw tokens from a probability distribution; temperature changes its concentration, and top-p/top-k can restrict candidates. This pilot does not sample. A future RL comparison must define its sampled evaluation separately.
- **0-shot / 5-shot:** zero or five worked examples in the prompt. Five-shot examples come from the subject's dev split, never its test answers.
- **Strict GSM8K extraction:** one final line such as `#### 72`. Units, percent signs, extra markers or non-final markers are invalid.
- **Flexible extraction:** also permits explicit final-answer phrases or a final number-only line; it does not mine intermediate reasoning for a number. Numeric comparisons are exact, including fractions and decimals.
- **Text versus logit scoring:** text scoring measures the answer the model generates; constrained scoring compares four next-token scores without generating a response. Constrained records therefore have `generated_text=null`.
- **Token cap:** GSM8K allows 1,024 new tokens; MMLU text allows 32. A capped output stays in the denominator and is incorrect, even if an answer appears.
- **Uncertainty range:** GSM8K uses a Wilson 95% interval; MMLU uses 2,000 bootstrap resamples within each subject. These ranges concern item sampling, not uncertainty over seeds or protocols.

## 1. Choose a GPU and name this attempt

In Colab, select **Runtime → Change runtime type → GPU**. Use one visible CUDA GPU and Python 3.10+.
Colab availability and session duration vary; no runtime estimate is promised.

The default baseline has no adapter. Keep the same `PLAN_NAME` to resume it.
Use a new name for a deliberate replicate or a change to precision, batching or adapter.
The default batch size is 1 to keep the first GPU attempt modest; it is recorded in the plan.
Model precision is chosen from GPU support during preparation and becomes fixed for that plan.

The harness commit below is pinned so opening this notebook later does not silently change the instrument.

In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys

REPO_URL = "https://github.com/mahadikprasad15/spar-pilot.git"
HARNESS_COMMIT = "55be867d9b0757dcd3c18b23352f644efeb062f3"
PLAN_NAME = "baseline-v1"  # change only for a new attempt/protocol variant
BATCH_SIZE = 1
ADAPTER_PATH = None  # baseline; later: a local PEFT directory or Hub repository
DRIVE_WORKSPACE = "SPAR/pilot1"
REPO_DIR = Path("/content/spar-pilot")

assert sys.version_info >= (3, 10), "Use Python 3.10 or later."
assert re.fullmatch(r"[A-Za-z0-9_-]+", PLAN_NAME), "Choose a simple plan name."
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

## 2. Mount Drive and restore the pinned repository

Authorize Google Drive access when Colab asks. Artifacts live in your Drive workspace,
exposed as the repository's `artifacts/` directory. Model/dataset caches remain on the temporary VM.
Saving the notebook alone does not save its VM files. Drive operations can fail or lag, so keep a final export too.

Reconnecting: rerun setup with the same plan. The saved dependency versions and frozen input files are reused.
The harness refuses a resume if recorded software or hardware differs; inspect that mismatch rather than deleting the config.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORKSPACE = Path("/content/drive/MyDrive") / DRIVE_WORKSPACE
ARTIFACT_ROOT = WORKSPACE / "artifacts"
SESSION_META = ARTIFACT_ROOT / "notebook" / PLAN_NAME
SESSION_META.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if current != HARNESS_COMMIT:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip()
    if dirty:
        raise RuntimeError("Repository has local edits. Preserve them before changing its revision.")
    subprocess.run(["git", "fetch", "origin", HARNESS_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", HARNESS_COMMIT], cwd=REPO_DIR, check=True)

link = REPO_DIR / "artifacts"
if link.is_symlink():
    assert link.resolve() == ARTIFACT_ROOT.resolve(), "Existing artifact link points elsewhere."
elif link.exists():
    raise RuntimeError("Existing local artifacts found. Preserve or restore them before attaching Drive.")
else:
    link.symlink_to(ARTIFACT_ROOT, target_is_directory=True)

os.chdir(REPO_DIR)
session_identity = {"harness_commit": HARNESS_COMMIT, "plan": PLAN_NAME,
                    "repo": REPO_URL, "artifact_root": str(ARTIFACT_ROOT)}
identity_file = SESSION_META / "identity.json"
if identity_file.exists():
    assert json.loads(identity_file.read_text()) == session_identity, "Saved notebook identity differs."
else:
    identity_file.write_text(json.dumps(session_identity, indent=2) + "\n")
print("Artifacts:", ARTIFACT_ROOT)
print("Harness commit:", HARNESS_COMMIT)

## 3. Install, verify the GPU, and run the CPU tests

On a new plan, install the package's GPU/test dependencies and record their exact resolved versions.
On resume, constrain installation to those saved core versions. If Colab requests a restart after installation,
restart the session and rerun sections 1–3. Each evaluation runs in a fresh Python subprocess.

The tests validate the measurement logic using fake external boundaries. Passing them does not establish GPU correctness
or the expected benchmark score. The first real-item audit is the next check.

In [ ]:
constraints = SESSION_META / "requirements-core.txt"
install = [sys.executable, "-m", "pip", "install", "-e", ".[gpu,test]"]
if constraints.exists():
    install += ["-c", str(constraints)]
subprocess.run(install, cwd=REPO_DIR, check=True)

import importlib.metadata as metadata
core = ["torch", "transformers", "datasets", "peft", "accelerate", "huggingface-hub", "pytest"]
resolved = "\n".join(f"{name}=={metadata.version(name)}" for name in core) + "\n"
if constraints.exists():
    assert constraints.read_text() == resolved, "Core dependency versions differ from the saved plan."
else:
    constraints.write_text(resolved)
(SESSION_META / "pip-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

gpu_probe = "import json, torch; print(json.dumps(dict(count=torch.cuda.device_count(), name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)))"
GPU = json.loads(subprocess.check_output([sys.executable, "-c", gpu_probe], text=True))
assert GPU["count"] == 1, "Select a Colab GPU runtime with exactly one visible GPU."
print("GPU:", GPU)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)

## 4. Freeze revisions, cohorts and the five cells

Seed 42 selects 150 GSM8K test indices and 20 per MMLU subject. A pinned dataset revision defines the index ordering.
Five-shot dev examples stay in published same-subject order. The same MMLU test IDs are reused across all four cells.
Preparation validates context length and the single-token A/B/C/D continuations; it fails instead of truncating or changing scorers.

Preparation downloads datasets and tokenizer/config files, not model weights. `bf16` is preferred when supported;
otherwise this notebook creates an `fp16` variant. A saved plan's precision is reused.

In [ ]:
PLAN_DIR = ARTIFACT_ROOT / "plans" / PLAN_NAME
pins_file = PLAN_DIR / "pins.json"
if pins_file.exists():
    saved_options = json.loads(pins_file.read_text())["options"]
    DTYPE = saved_options["dtype"]
else:
    DTYPE = "bfloat16" if GPU["bf16"] else "float16"

command = [sys.executable, "-m", "pilot_eval", "prepare", "--plan", PLAN_NAME,
           "--dtype", DTYPE, "--batch-size", str(BATCH_SIZE), "--output-root", str(ARTIFACT_ROOT)]
if ADAPTER_PATH:
    command += ["--adapter", ADAPTER_PATH]
subprocess.run(command, cwd=REPO_DIR, check=True)
plan_manifest = json.loads((PLAN_DIR / "manifest.json").read_text())
CONFIG_PATHS = [ARTIFACT_ROOT / name for name in plan_manifest["configs"]]
CONFIGS = [json.loads(path.read_text()) for path in CONFIG_PATHS]

for config in CONFIGS:
    items = json.loads((ARTIFACT_ROOT / config["items_path"]).read_text())
    print(f"{config['scorer']:12} {config['shots']}-shot | {len(items):4} items | {config['dtype']} | adapter={config['adapter']}")
print("Model revision:", CONFIGS[0]["model_revision"])

## 5. Inspect a rendered prompt before inference

Change `CELL_TO_PREVIEW` to see GSM8K, zero-shot MMLU or five-shot MMLU. Gold answers shown **below** the prompt
are evaluation metadata; they must not be present as the test answer inside the prompt. MMLU demonstrations do include their dev answers.
Both MMLU scorers share the assistant-side `Answer:` prefix. No system message is added.

In [ ]:
CELL_TO_PREVIEW = 2  # order: GSM8K, text-0, text-5, logits-0, logits-5
preview_config = CONFIGS[CELL_TO_PREVIEW]
preview_items = json.loads((ARTIFACT_ROOT / preview_config["items_path"]).read_text())
preview = preview_items[0]
print("ID:", preview["id"], "| prompt tokens:", preview["prompt_tokens"])
print("\n--- RENDERED PROMPT ---\n" + preview["prompt"])
print("\n--- EVALUATION METADATA (not appended to the prompt) ---")
print("Gold:", preview["gold"])
print("Dev source indices:", preview["dev_source_indices"])
print("Candidate token IDs:", preview.get("choice_token_ids"))

## 6. Run the fixed five-item audit for all cells

These are separately named `-audit-5` runs. MMLU's fixed prefix can cover only the first subject;
the audit checks pipeline behavior and is not a representative accuracy estimate.
The first inference downloads model weights. Each cell loads the model in its own subprocess,
then releases that process's GPU memory before the next cell.

Progress comes from durable status files. Every successful batch is saved. If a run stops, its console log and error log
are preserved. A tie/nonfinite logit makes an item invalid and stops the run. A cap is counted as incorrect.

In [ ]:
def run_folder(config, audit_items=None):
    run_id = config["run_id"] + (f"-audit-{audit_items}" if audit_items else "")
    return ARTIFACT_ROOT / "runs" / config["experiment"] / config["model"].replace("/", "--") / config["dataset"] / config["cohort"] / config["variant"] / run_id

def run_cli(config_path, audit_items=None):
    config = json.loads(config_path.read_text())
    run_dir = run_folder(config, audit_items)
    label = config["run_id"] + (f"-audit-{audit_items}" if audit_items else "")
    console = SESSION_META / f"{label}.console.log"
    command = [sys.executable, "-u", "-m", "pilot_eval", "run", "--config", str(config_path),
               "--output-root", str(ARTIFACT_ROOT)]
    if audit_items:
        command += ["--audit-items", str(audit_items)]
    print("Starting:", label, flush=True)
    last = None
    with console.open("a") as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    returncode = process.wait(timeout=10)
                    break
                except subprocess.TimeoutExpired:
                    status_file = run_dir / "meta/status.json"
                    if status_file.exists():
                        status = json.loads(status_file.read_text())
                        progress = (status["state"], status["completed"], status["total"])
                        if progress != last:
                            print(f"  {progress[0]}: {progress[1]}/{progress[2]}", flush=True)
                            last = progress
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        print("\n".join(console.read_text().splitlines()[-35:]))
        raise RuntimeError(f"Run stopped. Inspect {console} and {run_dir / 'logs/errors.jsonl'}")
    summary = json.loads((run_dir / "results/results.json").read_text())
    print("Completed:", label, "| items:", summary["total"], flush=True)
    return summary

AUDIT_SIZE = 5
AUDIT_SUMMARIES = [run_cli(path, AUDIT_SIZE) for path in CONFIG_PATHS]

## 7. Review the actual audit responses

For each of the 25 item records, inspect the rendered prompt, gold answer and output or four logits.
Check final answer extraction, contradictory choices, caps, stop reasons and token counts. The display selector below lets you visit each item.
For constrained scoring, `None` text/length/stop values are expected because no text was generated.
If anything looks wrong, stop and bring the saved record/error back for diagnosis before the full cohort.

In [ ]:
AUDIT_RECORDS = []
for config in CONFIGS:
    audit_dir = run_folder(config, AUDIT_SIZE)
    records = [json.loads(line) for line in (audit_dir / "results/responses.jsonl").read_text().splitlines()]
    for record in records:
        AUDIT_RECORDS.append((config, record))
    print(f"{config['scorer']} / {config['shots']}-shot: {len(records)} records; {audit_dir}")

AUDIT_ITEM_TO_REVIEW = 0  # rerun with 0..24 to inspect every record
config, record = AUDIT_RECORDS[AUDIT_ITEM_TO_REVIEW]
print("\nCell:", config["scorer"], config["shots"], "shot; ID:", record["id"])
print("\nPROMPT:\n" + record["prompt"])
print("\nGOLD:", record["gold"])
print("\nGENERATED TEXT:\n", record.get("generated_text"))
print("LOGITS:", record.get("candidate_scores"))
print("SCORE:", json.dumps(record["score"], indent=2))
print("TOKENS:", record.get("token_count"), "STOP:", record.get("stop_reason"))

## 8. Record the audit decision

After reviewing the prompts and outputs, set the checkbox to `True` and write a short observation.
This records a dated manual review tied to the exact configs and audit responses. It does not assert that an accuracy target was reached.
If you reconnect, the saved review may be reused only when those exact files still match.

In [ ]:
import hashlib
from datetime import datetime, timezone

I_HAVE_REVIEWED_THE_AUDIT = False  # @param {type:"boolean"}
AUDIT_NOTES = ""  # @param {type:"string"}

def audit_signature():
    digest = hashlib.sha256()
    for path, config in zip(CONFIG_PATHS, CONFIGS):
        digest.update(path.read_bytes())
        digest.update((run_folder(config, AUDIT_SIZE) / "results/responses.jsonl").read_bytes())
        digest.update((run_folder(config, AUDIT_SIZE) / "results/results.json").read_bytes())
    return digest.hexdigest()

review_file = SESSION_META / "audit-review.json"
if I_HAVE_REVIEWED_THE_AUDIT:
    assert AUDIT_NOTES.strip(), "Record a short observation from your audit."
    review = dict(approved=True, notes=AUDIT_NOTES, signature=audit_signature(),
                  reviewed_at=datetime.now(timezone.utc).isoformat(), harness_commit=HARNESS_COMMIT)
    review_file.write_text(json.dumps(review, indent=2) + "\n")
    print("Audit review saved.")
else:
    print("No new approval recorded. Review the audit, check the box and add notes before the first full run.")

## 9. Run the full cohorts (or resume them)

This cell verifies the saved audit review before starting. It runs GSM8K and all four MMLU settings separately.
Keep the browser session active as appropriate for Colab; this notebook does not attempt to bypass Colab runtime limits.
To split the work across sessions, set `CELLS_TO_RUN` to a subset of indices 0–4.

Repeat this cell with the same plan to resume. Completed runs are verified and do not regenerate;
partial runs process only missing IDs. Do not change batch size or precision under an existing plan.
An OOM or software/hardware mismatch needs inspection; a changed setup uses a new named plan.

In [ ]:
assert review_file.exists(), "Run and review the audit first (sections 6–8)."
review = json.loads(review_file.read_text())
assert review["approved"] and review["signature"] == audit_signature(), "Audit files/configs differ from the approved review."
CELLS_TO_RUN = [0, 1, 2, 3, 4]
FULL_SUMMARIES = {index: run_cli(CONFIG_PATHS[index]) for index in CELLS_TO_RUN}

## 10. Read results and uncertainty ranges

The table below contains full-cohort results only. Missing or incomplete runs are reported as such, without a provisional score.
Both GSM8K scorers use the same saved responses; MMLU has four distinct runs.
The historical gap is **measured score minus reference score**, not a correctness gate.
If strict and flexible scores differ, inspect answer formatting before attributing the gap to capability.
Response length is generated tokens before EOS, excluding EOS/padding, across all items including invalid/capped items.

For later adapters, compare the same item IDs and preserve precision, prompts, decoding and scorers.
Behavioral differences alone do not establish where the adapter wrote or why an effect occurred.

In [ ]:
from IPython.display import display, Markdown

rows = []
for config in CONFIGS:
    directory = run_folder(config)
    result_file = directory / "results/results.json"
    if not result_file.exists():
        status_file = directory / "meta/status.json"
        status = json.loads(status_file.read_text()) if status_file.exists() else {"state": "not started"}
        print(config["scorer"], config["shots"], "shot:", status)
        continue
    result = json.loads(result_file.read_text())
    if config["scorer"] == "gsm8k":
        for scorer in ("strict", "flexible"):
            rows.append((f"GSM8K {scorer}", result[f"{scorer}_accuracy"], result[f"{scorer}_wilson_95"],
                         result[f"{scorer}_invalid_rate"], result["cap_rate"], result["historical_reference"][f"{scorer}_accuracy_gap"]))
        print("GSM8K tokens: mean", result["mean_response_tokens"], "median", result["median_response_tokens"],
              "mean gap vs 288:", result["historical_reference"]["mean_response_tokens_gap"])
    else:
        label = f"MMLU {config['scorer']} {config['shots']}-shot" + (" (primary)" if config["scorer"] == "mmlu_logits" and config["shots"] == 5 else "")
        rows.append((label, result["accuracy"], result["sampling_interval_95"], result["invalid_rate"],
                     result["cap_rate"], result["historical_reference"]["accuracy_gap"]))

table = "| Metric | Accuracy | 95% range | Invalid | Capped | Gap vs reference |\n|---|---:|---|---:|---:|---:|\n"
for label, score, interval, invalid, capped, gap in rows:
    table += f"| {label} | {score:.3f} | [{interval[0]:.3f}, {interval[1]:.3f}] | {invalid:.1%} | {capped:.1%} | {gap:+.3f} |\n"
display(Markdown(table))

## 11. Save your interpretation and export the evidence

Write observed numbers, format/cap issues, and differences from the reference. Keep possible explanations separate from findings.
The original artifacts already live on Drive. The optional ZIP downloads a copy of **all artifacts** in this workspace
(including other plans); it excludes model caches. Keep configs, raw responses, cohort manifests and errors, not only the summary table.

In [ ]:
OBSERVATIONS = ""  # @param {type:"string"}
if OBSERVATIONS.strip():
    note = dict(plan=PLAN_NAME, observations=OBSERVATIONS, harness_commit=HARNESS_COMMIT,
                recorded_at=datetime.now(timezone.utc).isoformat())
    filename = "observations-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json"
    (SESSION_META / filename).write_text(json.dumps(note, indent=2) + "\n")
print("Durable artifact root:", ARTIFACT_ROOT)

DOWNLOAD_ZIP = False  # @param {type:"boolean"}
if DOWNLOAD_ZIP:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(Path("/content") / f"{PLAN_NAME}-artifacts"), "zip", root_dir=ARTIFACT_ROOT)
    files.download(archive)

## Troubleshooting and next steps

- **No GPU:** select a GPU runtime, reconnect and rerun setup.
- **Installation/version error:** preserve the traceback and core version file. Resume requires the saved environment; do not remove pins to force a run.
- **Out of memory:** retained batches remain saved. For a fresh attempt, choose a new plan with smaller batch size. Do not change an immutable run config.
- **Choice token validation, exact tie, nonfinite logits, corrupt JSONL:** stop and inspect the specific item/error. The harness will not silently switch scorer or drop records.
- **Drive I/O failure:** stop and check storage/mount availability. Saving the notebook is not sufficient to save evaluation outputs.
- **Low accuracy or changed length:** inspect prompts, formats, caps and precision. Report the measured gap; it is not automatically a reproduction failure.

After a clean baseline: freeze a dated prediction before training, separate held-out test IDs from training,
train matched SFT/GRPO adapters under the B0 protocol, evaluate with the same instrument, then move to effective-write and causal tests.
This notebook supports one optional PEFT adapter. A later adapter plan should explicitly reuse baseline revisions and IDs;
newly resolving Hub revisions alone does not guarantee a paired comparison.

**Protocol:** [spec](https://github.com/mahadikprasad15/spar-pilot/blob/55be867d9b0757dcd3c18b23352f644efeb062f3/.scratch/pilot-1-evaluation/spec.md)
and [README](https://github.com/mahadikprasad15/spar-pilot/blob/55be867d9b0757dcd3c18b23352f644efeb062f3/README.md).
Colab storage, mounting and runtime behavior: [official FAQ](https://research.google.com/colaboratory/faq.html).